# Шаг 1. Первые четыре страницы PDF → pandas DataFrame

Разбираем пояснения к расписанию. Рейсы со страницы 5 и далее пока не читаем. Изображения учитываем отдельно, OCR не выполняем. Исходный PDF не изменяется.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "parse_intro.py").is_file():
    ROOT = ROOT / "parsers" / "PDF_Parser"
if not (ROOT / "parse_intro.py").is_file():
    raise FileNotFoundError(
        "Откройте notebook из корня проекта или папки parsers/PDF_Parserr"
    )
sys.path.insert(0, str(ROOT))

In [2]:
from parse_intro import parse_intro, save_frames

PDF = ROOT.parents[1] / "raw_data/Skyteam_Timetable.pdf"
frames = parse_intro(PDF)
{table: len(df) for table, df in frames.items()}

{'raw_lines': 167, 'pages': 4, 'images': 9, 'metadata': 4, 'contacts': 20, 'sections': 2, 'field_definitions': 9, 'weekdays': 7, 'aircraft_types': 82}

## Что считается строкой

`raw_lines`: одна непустая текстовая строка PDF со своей страницей и координатами. Остальные таблицы — производные объекты. `source_record_ids` связывает каждую такую строку со строками `raw_lines`; это не наблюдения о перелётах.

In [3]:
raw_df = frames["raw_lines"]
raw_df[["page", "x0", "y0", "raw_text"]].head(12)

    page  ...                                        raw_text
0      1  ...                               SkyTeam Timetable
1      1  ...  Covers period: 01 Nov 2018 through 31 Jan 2019
2      1  ...                             Worldwide timetable
3      2  ...                                         Contact
4      2  ...    To book, contact any SkyTeam member airline.
5      2  ...                      Aeroflot  www.aeroflot.com
6      2  ...        Aerolneas Argentinas  www.aerolineas.com
7      2  ...                  Aeromexico  www.aeromexico.com
8      2  ...                   Air Europa  www.aireuropa.com
9      2  ...                   Air France  www.airfrance.com
10     2  ...                      Alitalia  www.alitalia.com
11     2  ...          China Airlines  www.china-airlines.com

[12 rows x 4 columns]

## Обложка и описание источника

Даты преобразуются только по явному английскому названию месяца. Оговорки о полноте и актуальности сохраняются дословно в `sections`.

In [4]:
metadata_df = frames["metadata"]
metadata_df[["key", "raw_value", "value", "page"]]

              key  ... page
0      cover_text  ...    1
1  coverage_start  ...    1
2    coverage_end  ...    1
3      cover_text  ...    1

[4 rows x 4 columns]

In [5]:
contacts_df = frames["contacts"]
contacts_df[["airline_name_raw", "website_raw", "page"]]

        airline_name_raw               website_raw  page
0               Aeroflot          www.aeroflot.com     2
1   Aerolneas Argentinas        www.aerolineas.com     2
2             Aeromexico        www.aeromexico.com     2
3             Air Europa         www.aireuropa.com     2
4             Air France         www.airfrance.com     2
5               Alitalia          www.alitalia.com     2
6         China Airlines    www.china-airlines.com     2
7          China Eastern             www.ceair.com     2
8         China Southern             www.csair.com     2
9         Czech Airlines     www.czechairlines.com     2
10       Delta Air Lines             www.delta.com     2
11      Garuda Indonesia  www.garuda-indonesia.com     2
12         Kenya Airways     www.kenya-airways.com     2
13                   KLM               www.klm.com     2
14            Korean Air         www.koreanair.com     2
15  Middle East Airlines            www.mea.com.lb     2
16                Saudia     ww

In [6]:
for row in frames["sections"].itertuples():
    print(row.section, "\n", row.text_raw, "\n")

disclaimer 
 The content of this PDF timetable is for information
purposes only, subject to change at any time. Neither
SkyTeam, nor SkyTeam Members (including without
limitation their respective suppliers) make representation
or give warranty as to the completeness or accuracy of
such content as well as to its suitability for any purpose.
In particular, you should be aware that this content may
be incomplete, may contain errors or may have become
out of date. It is provided as is without any warranty or
condition of any kind, either express or implied, including
but not limited to all implied warranties and conditions of
merchantability, fitness for a particular purpose, title and
non-infringement. Given the flexible nature of flight
schedules, our PDF timetable may not reflect the latest
information.
By accessing the PDF timetable, the user acknowledges
that the SkyTeam Alliance and any SkyTeam member
airline will not be responsible or liable to the user, or any
other party, for any 

## Словарь полей расписания

День 1 означает воскресенье, день 7 — субботу. DepTime и ArrTime — местное время, не UTC. `+1` обозначает прибытие на следующий день. Эти определения извлечены из документа; часовые пояса не назначаются.

In [7]:
fields_df = frames["field_definitions"]
fields_df[["field", "definition_raw", "page"]]

         field                                     definition_raw  page
0         From  Departure airport details (city name, country ...     3
1           To  Arrival airport details (city name, country na...     3
2     Validity                The flight operates in period shown     3
3         Days  1 - Sunday, 2 - Monday, 3 - Tuesday, 4 - Wedne...     3
4      DepTime                     Flight departure in local time     3
5      ArrTime  Flight arrival in local time. +1 means next-da...     3
6       Flight                                      Flight number     3
7     Aircraft      Aircraft type. See next page for full details     3
8  Travel Time  Total travel time from departure airport to ar...     3

In [8]:
weekdays_df = frames["weekdays"]
weekdays_df[["day_code", "weekday_name"]]

  day_code weekday_name
0        1       Sunday
1        2       Monday
2        3      Tuesday
3        4    Wednesday
4        5     Thursday
5        6       Friday
6        7     Saturday

## Справочник оборудования

Коды остаются строками. Описания сохраняются без исправлений. BUS и EQV не являются названиями конкретных моделей самолётов. Назначение кода не доказывает использование оборудования на реальном рейсе.

In [9]:
aircraft_df = frames["aircraft_types"]
aircraft_df[["aircraft_code", "description_raw", "column", "page"]]

   aircraft_code                      description_raw column  page
0            100                           Fokker 100   left     4
1            310      Airbus Industrie A310 Passenger  right     4
2            313  Airbus Industrie A310-300 Passenger   left     4
3            318                Airbus Industrie A318  right     4
4            319                Airbus Industrie A319   left     4
..           ...                                  ...    ...   ...
77           M88               Boeing (Douglas) MD-88  right     4
78           M90               Boeing (Douglas) MD-90   left     4
79           S20                            Saab 2000  right     4
80           SF3                             Saab 340   left     4
81           SU9                      Superjet 100-95  right     4

[82 rows x 4 columns]

## Изображения и границы результата

Иллюстративная таблица маршрута на странице 3 находится в растровом изображении. Она не включена в таблицу реальных рейсов. Логотипы тоже не транскрибируются. Сохранены координаты изображений.

Статус `not_independently_verified` означает отсутствие независимой проверки достоверности источника. Визуальная сверка и проверка сохранения типов не доказывают историческую точность расписания.

In [10]:
frames["images"][["page", "image_index", "bbox_json", "parse_status"]]

   page  ...           parse_status
0     1  ...  image_not_transcribed
1     2  ...  image_not_transcribed
2     2  ...  image_not_transcribed
3     2  ...  image_not_transcribed
4     3  ...  image_not_transcribed
5     3  ...  image_not_transcribed
6     3  ...  image_not_transcribed
7     4  ...  image_not_transcribed
8     4  ...  image_not_transcribed

[9 rows x 4 columns]

## Сохранение

Parquet сохраняет типы, CSV предназначен для просмотра. При чтении CSV коды нужно явно читать строками. Запись Parquet проверяется обратным чтением и сравнением значений/типов.

In [11]:
OUTPUT = ROOT / "data/pdf_intro"
save_frames(frames, OUTPUT)
print(OUTPUT)

/Users/kharitonovda/Documents/ChatGPT/Spies_project/parsers/PDF_Parser/data/pdf_intro
